# Clase 2 · Práctica: Visión por computador I — redes convolucionales

## Qué vas a practicar

En esta libreta aplicas lo que viste en la Clase 2 sobre redes neuronales convolucionales (CNN). Completas el código en dos ejercicios:

- **Ejercicio 1.** Construyes la arquitectura de una CNN para clasificar prendas de ropa en el conjunto de datos FashionMNIST y la entrenas de principio a fin.
- **Ejercicio 2.** Reutilizas el modelo entrenado para inspeccionar visualmente sus errores: identificas qué imágenes clasifica mal y las muestras en una cuadrícula.

Cada ejercicio incluye el andamiaje de código con partes marcadas como `# TODO`, que debes completar tú. Debajo de cada ejercicio encuentras una solución colapsada: intenta resolver el ejercicio por tu cuenta antes de abrirla.

## Requisito previo

Necesitas haber visto la Clase 2 ("Visión por computador I: redes convolucionales"), en la que se explica qué es un filtro convolucional, cómo funciona el *pooling* y cómo se combinan ambos en una arquitectura CNN completa.

## Cómo ejecutar esta libreta

Ábrela en Google Colab y ejecuta las celdas en orden, de arriba a abajo. No instales nada adicional: `torch` y `torchvision` ya vienen preinstalados en Colab. Si tienes una GPU (procesador gráfico) asignada, el entrenamiento es más rápido, pero también funciona en CPU.

## Ejercicio 1 · Clasificador de prendas de ropa para una tienda online

### El caso

Trabajas en el equipo de datos de una tienda online de ropa. Cuando una persona vendedora sube una fotografía de una prenda al catálogo, quieres asignarle automáticamente una categoría (camiseta, pantalón, abrigo, bolso...) para agilizar la publicación y mantener el catálogo ordenado. Como primera aproximación, entrenas un clasificador con el conjunto de datos FashionMNIST, que contiene imágenes en escala de grises de 28 x 28 píxeles repartidas en diez categorías de prendas, y que sirve de sustituto razonable de las fotografías reales de tu catálogo.

### Lo que tienes que hacer

Completa la arquitectura de la red en la clase `CNNRopa`: el número y tamaño de las capas convolucionales, la capa de *pooling* y la capa lineal final. El bucle de entrenamiento y la función de evaluación ya están escritos: no los toques, solo tienes que conseguir que el modelo se defina y calcule correctamente hacia delante (`forward`).

### Pistas

- La imagen de entrada tiene forma `(lote, 1, 28, 28)`: un solo canal porque está en escala de grises.
- Usa dos bloques convolucionales. El primero recibe 1 canal de entrada y produce 32 canales de salida; el segundo recibe esos 32 canales y produce 64. En ambos, usa un filtro (*kernel*) de 3 x 3 con `padding=1`, para que la convolución sea "igual" (*same*) y no reduzca el tamaño espacial en ese paso.
- Aplica una capa de *max pooling* de 2 x 2 después de cada bloque convolucional. Cada *pooling* reduce el alto y el ancho a la mitad: 28 → 14 → 7.
- Tras los dos bloques y sus dos *pooling*, el mapa de activaciones tiene forma `(64, 7, 7)`. Al aplanarlo obtienes un vector de `64 * 7 * 7` valores: ese es el tamaño de entrada de la primera capa lineal.
- La capa lineal final debe producir tantas salidas (logits) como clases: diez, una por cada tipo de prenda. No le apliques `softmax` explícitamente: `nn.CrossEntropyLoss` ya lo incluye internamente.
- No olvides aplicar la no linealidad ReLU (unidad lineal rectificada) después de cada convolución y de la primera capa lineal, tal como viste en la Clase 2.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# Normalizamos con la media y desviación típica habituales de FashionMNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

clases = [
    "camiseta", "pantalón", "jersey", "vestido", "abrigo",
    "sandalia", "camisa", "zapatilla", "bolso", "botín",
]

In [ ]:
class CNNRopa(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()

        # TODO: define la primera capa convolucional.
        # Recibe 1 canal de entrada (escala de grises) y produce 32 canales
        # de salida. Usa kernel de 3x3 con padding=1 (convolución "igual").
        self.conv1 = ...

        # TODO: define la segunda capa convolucional.
        # Recibe los 32 canales de conv1 y produce 64 canales de salida.
        # Mismo kernel 3x3 y padding=1.
        self.conv2 = ...

        # TODO: define la capa de max pooling.
        # Usa una ventana de 2x2 que reduzca el alto y el ancho a la mitad.
        self.pool = ...

        # Tras aplicar pool dos veces sobre una entrada de 28x28: 28 -> 14 -> 7
        # El último mapa de activaciones tiene forma (64 canales, 7, 7)

        # TODO: define la primera capa totalmente conectada.
        # Recibe el vector aplanado (64 * 7 * 7 valores) y produce 128 salidas.
        self.fc1 = ...

        # TODO: define la capa final.
        # Recibe las 128 salidas de fc1 y produce num_clases salidas (los logits).
        self.fc2 = ...

        self.dropout = nn.Dropout(0.25)  # mitiga el sobreajuste

    def forward(self, x):
        # x llega con forma (lote, 1, 28, 28)

        # TODO: aplica conv1, después ReLU, después el pooling
        x = ...  # -> (lote, 32, 14, 14)

        # TODO: aplica conv2, después ReLU, después el pooling
        x = ...  # -> (lote, 64, 7, 7)

        x = torch.flatten(x, start_dim=1)  # aplana todo menos el eje de lote

        # TODO: aplica fc1 seguido de ReLU
        x = ...

        x = self.dropout(x)

        # TODO: aplica fc2 para obtener los logits (sin activación)
        x = ...

        return x


modelo = CNNRopa(num_clases=10).to(device)
print(modelo)

<details>
<summary>💡 Ver solución</summary>

```python
class CNNRopa(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()

        # Bloque convolucional 1: 1 canal de entrada -> 32 filtros
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32,
                                kernel_size=3, stride=1, padding=1)

        # Bloque convolucional 2: 32 -> 64 filtros
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64,
                                kernel_size=3, stride=1, padding=1)

        # Max pooling 2x2: reduce el alto y el ancho a la mitad cada vez
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Tras dos pooling de 2x2 sobre una entrada de 28x28: 28 -> 14 -> 7
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_clases)

        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> (lote, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # -> (lote, 64, 7, 7)

        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # logits, sin softmax explícito
        return x


modelo = CNNRopa(num_clases=10).to(device)
print(modelo)
```
</details>

### Entrena y evalúa el modelo

Esta parte ya está resuelta: define la función de pérdida, el optimizador, el bucle de entrenamiento y la evaluación, y entrena el modelo durante unas pocas épocas. Solo necesitas ejecutar las celdas. Si tu arquitectura del apartado anterior es correcta, el modelo entrena sin errores y alcanza una exactitud en torno al 90 % sobre el conjunto de prueba.

In [ ]:
# CrossEntropyLoss combina softmax + entropía cruzada en un solo paso
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


def evaluar(modelo, loader, criterio):
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida_total += criterio(salidas, etiquetas).item() * imagenes.size(0)
            predicciones = salidas.argmax(dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
    perdida_media = perdida_total / len(loader.dataset)
    exactitud = aciertos / len(loader.dataset)
    return perdida_media, exactitud


num_epocas = 5
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    perdida_test, exactitud_test = evaluar(modelo, test_loader, criterio)
    print(f"Época {epoca}/{num_epocas} | "
          f"pérdida train: {perdida_train:.4f} | "
          f"pérdida test: {perdida_test:.4f} | "
          f"exactitud test: {exactitud_test:.2%}")

print(f"\nExactitud final sobre el conjunto de test: {exactitud_test:.2%}")

## Ejercicio 2 · Inspecciona los errores del clasificador

### El caso

Tu modelo alcanza una buena exactitud en el conjunto de prueba, pero un 90 % de aciertos también significa que se equivoca en una de cada diez prendas. Antes de llevar este clasificador a producción en la tienda online, el equipo quiere ver ejemplos concretos de esos errores: ¿confunde camisetas con camisas? ¿Botines con zapatillas? Mirar los fallos con tus propios ojos ayuda a decidir si el error es razonable (prendas visualmente parecidas) o si señala un problema más profundo del modelo.

### Lo que tienes que hacer

Completa la función `obtener_ejemplos_mal_clasificados`, que recorre el conjunto de prueba con el modelo ya entrenado del Ejercicio 1 y guarda ejemplos en los que la predicción no coincide con la etiqueta real. La función `mostrar_cuadricula_errores`, que dibuja esos ejemplos en una cuadrícula con matplotlib, ya está resuelta: no la modifiques.

### Pistas

- `salidas.argmax(dim=1)` te da la clase predicha para cada imagen del lote.
- Compara `predicciones` con `etiquetas` elemento a elemento para obtener una máscara booleana de qué posiciones del lote están mal clasificadas.
- Recorre las posiciones del lote donde la máscara vale verdadero y añade a la lista `errores` una tupla `(imagen, etiqueta_real, etiqueta_predicha)` por cada una. Recuerda mover la imagen a CPU (`.cpu()`) antes de guardarla, porque matplotlib no trabaja con tensores en GPU.
- Detén la recogida en cuanto la lista `errores` alcance `max_ejemplos`, para no acumular más ejemplos de los que vas a mostrar.

In [ ]:
import matplotlib.pyplot as plt


def obtener_ejemplos_mal_clasificados(modelo, dataset, device, max_ejemplos=12):
    """Recorre el dataset y devuelve una lista de tuplas
    (imagen, etiqueta_real, etiqueta_predicha) para los ejemplos
    donde el modelo se equivoca."""
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    errores = []

    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes_dev = imagenes.to(device)
            etiquetas_dev = etiquetas.to(device)
            salidas = modelo(imagenes_dev)
            predicciones = salidas.argmax(dim=1)

            # TODO: calcula una máscara booleana (forma (batch,)) que sea
            # verdadera en las posiciones donde la predicción NO coincide
            # con la etiqueta real
            mascara_error = ...

            # TODO: recorre las posiciones del lote y, para cada una donde
            # mascara_error sea verdadera, añade a `errores` la tupla
            # (imagen, etiqueta_real, etiqueta_predicha). Usa
            # imagenes[i].cpu() para la imagen y .item() para convertir
            # las etiquetas a enteros de Python. Para en cuanto
            # len(errores) alcance max_ejemplos.
            ...

            if len(errores) >= max_ejemplos:
                break

    return errores[:max_ejemplos]


def mostrar_cuadricula_errores(errores, clases, columnas=4):
    """Dibuja una cuadrícula con las imágenes mal clasificadas,
    indicando la etiqueta real y la predicha en el título de cada una."""
    filas = (len(errores) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * 2.2, filas * 2.4))
    ejes = ejes.flatten()

    for ax, (imagen, real, predicha) in zip(ejes, errores):
        ax.imshow(imagen.squeeze(), cmap="gray")
        ax.set_title(f"real: {clases[real]}\npred: {clases[predicha]}", fontsize=9)
        ax.axis("off")

    for ax in ejes[len(errores):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


errores = obtener_ejemplos_mal_clasificados(modelo, test_dataset, device, max_ejemplos=12)
print(f"Ejemplos mal clasificados encontrados: {len(errores)}")
mostrar_cuadricula_errores(errores, clases)

<details>
<summary>💡 Ver solución</summary>

```python
def obtener_ejemplos_mal_clasificados(modelo, dataset, device, max_ejemplos=12):
    """Recorre el dataset y devuelve una lista de tuplas
    (imagen, etiqueta_real, etiqueta_predicha) para los ejemplos
    donde el modelo se equivoca."""
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    errores = []

    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes_dev = imagenes.to(device)
            etiquetas_dev = etiquetas.to(device)
            salidas = modelo(imagenes_dev)
            predicciones = salidas.argmax(dim=1)

            # Máscara booleana: verdadera donde la predicción falla
            mascara_error = predicciones != etiquetas_dev

            # Recorremos el lote y guardamos cada ejemplo mal clasificado
            indices_error = mascara_error.nonzero(as_tuple=True)[0]
            for i in indices_error:
                i = i.item()
                errores.append((
                    imagenes[i].cpu(),
                    etiquetas[i].item(),
                    predicciones[i].item(),
                ))
                if len(errores) >= max_ejemplos:
                    break

            if len(errores) >= max_ejemplos:
                break

    return errores[:max_ejemplos]


errores = obtener_ejemplos_mal_clasificados(modelo, test_dataset, device, max_ejemplos=12)
print(f"Ejemplos mal clasificados encontrados: {len(errores)}")
mostrar_cuadricula_errores(errores, clases)
```
</details>

## Para terminar

Ya has construido, entrenado y auditado una red neuronal convolucional de principio a fin: definiste su arquitectura, comprobaste su exactitud sobre datos que nunca había visto y miraste de cerca dónde falla. Ese ciclo completo (diseñar, entrenar, evaluar, inspeccionar errores) es exactamente el que vas a repetir con arquitecturas más sofisticadas en las próximas clases.

Antes de continuar, prueba a cambiar el número de filtros de `conv1` y `conv2` en el Ejercicio 1, o sustituye `nn.MaxPool2d` por `nn.AvgPool2d`, y observa cómo cambian la exactitud y los errores que ves en el Ejercicio 2.

**¡Comparte tus resultados y tus dudas en el canal del curso antes de la próxima clase en vivo!**